## Import Required Libraries

In [2]:
import sys
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

# Add parent directory to path to access document_summarization package
sys.path.append(os.path.abspath('../..'))

from document_summarization.summarization_service import SummarizationService
from document_summarization.utils.entity_loader import EntityLoader

print("✓ Modules imported successfully")

✓ Modules imported successfully


## Configuration

Load Azure OpenAI credentials from environment variables.

In [4]:
# Load environment variables
load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("GPT_4_1_API_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("GPT_4_1_API_DEPLOYMENT")
API_VERSION = os.getenv("GPT_4_1_API_VERSION")

print("✓ Configuration loaded")

✓ Configuration loaded


## Load Entity Data

Use the EntityLoader utility to load and extract entities from JSON.

In [5]:
# Load and extract entities in one step
data_path = os.path.join("data", "labels.json")
entities = EntityLoader.load_and_extract(data_path)

# Display entities
print(f"\\n{'='*80}")
print("EXTRACTED ENTITIES")
print(f"{'='*80}")
for name, value in entities.items():
    print(f"{name:<45}: {value}")
print(f"\\nTotal: {len(entities)} entities")

\n================================================================================
EXTRACTED ENTITIES
Applicant Name                               : Ms LOK WING CHING
Job Title of Applicant                       : DIRECTOR
Business Registration Number of Employer     : 21893829
Height of Applicant                          : 174 cm
Weight of Applicant                          : 77 kg
\nTotal: 5 entities


## Initialize Summarization Service

Create the service with Azure OpenAI configuration.

In [7]:
# Initialize the service
service = SummarizationService(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    deployment_name=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    credential=DefaultAzureCredential(),
    temperature=0.0,
    max_tokens=5000
)

print("✓ SummarizationService initialized")

✓ SummarizationService initialized


## Generate Summary

Use the service to generate a natural language summary.

In [8]:
# Generate summary
result = service.generate_summary(
    entities=entities,
    context="Insurance Application Form"
)

# Display results
print(f"\\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")

if result["success"]:
    print(f"\\n{result['summary']}")
    print(f"\\n{'='*80}")
    print("\\nMetadata:")
    for key, value in result["metadata"].items():
        print(f"  {key}: {value}")
else:
    print(f"\\n⚠ Error: {result['error_message']}")

\n================================================================================
SUMMARY
\nThe applicant, Ms. Lok Wing Ching, is currently employed as a Director. Her employer’s business registration number is 21893829. Ms. Lok’s height is recorded as 174 cm and her weight is 77 kg.
\n================================================================================
\nMetadata:
  entity_count: 5
  summary_length: 192
  summary_words: 33
  temperature: 0.0
  model: gpt-4.1
  prompt_tokens: 204
  completion_tokens: 49
  total_tokens: 253


## Save Results

Save the summary and entities to output files.

In [9]:
# Save results
if result["success"]:
    file_paths = service.save_summary(
        summary=result["summary"],
        entities=entities,
        metadata=result["metadata"],
        output_dir="output"
    )
    
    print("\\n✓ Results saved:")
    print(f"  Text: {file_paths['summary_file']}")
    print(f"  JSON: {file_paths['json_file']}")
    print(f"\\n📁 Location: {os.path.abspath('output')}")
else:
    print("⚠ No summary to save")

\n✓ Results saved:
  Text: output/application_summary.txt
  JSON: output/summarization_results.json
\n📁 Location: /Users/anishganguli/Documents/Projects/HSBC/PoC/HSBC_IWPB_UW/src/document_summarization/notebooks/output


## Export as JSON

Convert the result to JSON format for API responses or further processing.

In [10]:
# Export to JSON
json_output = service.to_json(result, indent=2)

print(f"\\n{'='*80}")
print("JSON OUTPUT")
print(f"{'='*80}")
print(json_output[:500] + "..." if len(json_output) > 500 else json_output)

\n================================================================================
JSON OUTPUT
{
  "summary": "The applicant, Ms. Lok Wing Ching, is currently employed as a Director. Her employer’s business registration number is 21893829. Ms. Lok’s height is recorded as 174 cm and her weight is 77 kg.",
  "success": true,
  "error_message": null,
  "metadata": {
    "entity_count": 5,
    "summary_length": 192,
    "summary_words": 33,
    "temperature": 0.0,
    "model": "gpt-4.1",
    "prompt_tokens": 204,
    "completion_tokens": 49,
    "total_tokens": 253
  }
}
